# Marketing Campaign Data

This notebook is a direct implementation of the YouTube video entitled [Watch me Cleaning Data in minutes with Python](https://youtu.be/NeJKaolLQqU?si=fODZ3egk5a-zm9vX) by **Lore So What**.

## Imports and Loading

In [149]:
import numpy as np
import pandas as pd

In [150]:
df = pd.read_csv("data/raw.csv")

print(f"Loaded Dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Loaded Dataset: 2020 rows, 12 columns


In [151]:
df.head()

,Campaign_ID,Campaign_Name,Start_Date,End_Date,Channel,Impressions,Clicks,Spend,Conversions,Active,Clicks,Campaign_Tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24 00:00:00,2023-12-13,TikTok,16795,197,$102.82,20.0,Y,NaN,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06 00:00:00,2023-05-12,Facebook,1860,30,24.33,1.0,0,NaN,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13 00:00:00,2023-12-20,Email,77820,843,1323.39,51.0,No,NaN,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,2023-10-30,2023-11-03,TikTok,55886,2019,2180.38,135.0,True,NaN,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22 00:00:00,2023-04-23,Facebook,7265,169,252.44,30.0,Yes,NaN,FA


In [152]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2020 entries, 0 to 2019
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0    Campaign_ID   2020 non-null   str    
 1   Campaign_Name  2020 non-null   str    
 2   Start_Date     2020 non-null   str    
 3   End_Date       2020 non-null   str    
 4   Channel        1919 non-null   str    
 5   Impressions    2020 non-null   int64  
 6   Clicks         2020 non-null   int64  
 7   Spend          2020 non-null   str    
 8   Conversions    1820 non-null   float64
 9   Active         2020 non-null   str    
 10  Clicks         40 non-null     float64
 11  Campaign_Tag   2020 non-null   str    
dtypes: float64(2), int64(2), str(8)
memory usage: 189.5 KB


In [153]:
row_inspect = df.iloc[3]

## Step 1: Header Cleaning

`df.columns.str` provides an accessor to apply string methods to all column names in the pandas DataFrame simultaneously.

In [154]:
print(df.columns.to_list())

df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print(df.columns.to_list())


[' Campaign_ID ', 'Campaign_Name', 'Start_Date', 'End_Date', 'Channel', 'Impressions', 'Clicks ', 'Spend', 'Conversions', 'Active', 'Clicks', 'Campaign_Tag']
['campaign_id', 'campaign_name', 'start_date', 'end_date', 'channel', 'impressions', 'clicks', 'spend', 'conversions', 'active', 'clicks', 'campaign_tag']


## Step 2: Type Conversion and Currency Cleaning

By default, pandas `str.contains()` treats the search string as a regular expression (regex).

In regular expressions, the \$ symbol has a special meaning: it represents the end of a string. By writing `r"\$"`, we tell the regex engine to treat \$ as a literal dollar sign character.

Prefixing a string with r in Python ensures that backslashes are treated as literal backslashes and not converted into Python escape sequences (like `\n` or `\t`).

An alternative approach to this is:
```
df["spend"].str.contains("$", regex=False)
```

In [155]:
dirty_spend_mask = df["spend"].str.contains(r"\$")

print(f"Total records with a dollar sign in spend: {dirty_spend_mask.sum()}")
display(df.loc[dirty_spend_mask, ["campaign_id", "spend"]].head())

Total records with a dollar sign in spend: 298


,campaign_id,spend
0,CMP-00001,$102.82
21,CMP-00022,$2428.4
22,CMP-00023,$4726.22
31,CMP-00032,$2759.35
32,CMP-00033,$2393.02


In data cleaning (and specifically within pandas and NumPy), a "mask" refers to a sequence or array of Boolean values (True and False) used to filter, select, or modify data.

In [156]:
df["spend"] = df["spend"].str.replace(r"[^\d.-]", "", regex=True)

df["spend"] = pd.to_numeric(df["spend"])

The search regex `[^\d.-]` means *any character that is not a digit, a dot, or a minus sign*. We're basically replacing anything that is not one of the three mentioned characters with nothing, which eliminates the dollar sign and any other anomally.

In [157]:
display(df.loc[dirty_spend_mask, ["campaign_id", "spend"]].head())

,campaign_id,spend
0,CMP-00001,102.82
21,CMP-00022,2428.40
22,CMP-00023,4726.22
31,CMP-00032,2759.35
32,CMP-00033,2393.02


## Step 3: Categorical Typos (Fuzzy Logic)

In [158]:
print(df["channel"].unique().tolist())

channel_map = {
    "E-mail": "Email",
    "Gogle": "Google Ads",
    "Tik_Tok": "TikTok",
    "Facebok": "Facebook",
    "Insta_gram": "Instagram"
}

df["channel"] = df["channel"].replace(channel_map)

print(df["channel"].unique().tolist())

['TikTok', 'Facebook', 'Email', 'Instagram', 'Google Ads', 'E-mail', nan, 'Gogle', 'Tik_Tok', 'Facebok', 'Insta_gram']
['TikTok', 'Facebook', 'Email', 'Instagram', 'Google Ads', nan]


## Step 4: Handling Mixed Booleans

In [159]:
print(df["active"].unique().tolist())

active_map = {
    "Y": True,
    "True": True,
    "Yes": True,
    "1": True,
    "False": False,
    "No": False,
    "0": False,
}

df["active"] = df["active"].replace(active_map).fillna(False).astype(bool)

print(df["active"].unique().tolist())

['Y', '0', 'No', 'True', 'Yes', '1', 'False']
[True, False]


## Step 5: Date Parsing

- `errors="coerce"` to handle unconvertable data as NaT (Not a Time)
- To tell pandas that the column contains varying formats and that it needs to evaluate them individually, you should use the `format="mixed"` argument (available in Pandas 2.0+). This will force pandas to figure out the format row-by-row, successfully parsing both MM/DD/YYYY and YYYY-MM-DD within the YYYY-MM-DD entries.

In [160]:
print(df["start_date"].dtype)
print(df["end_date"].dtype)

df["start_date"] = pd.to_datetime(df["start_date"], errors="coerce", format="mixed")
df["end_date"] = pd.to_datetime(df["end_date"], errors="coerce", format="mixed")

print(df["start_date"].dtype)
print(df["end_date"].dtype)

str
str
datetime64[us]
datetime64[us]


## Step 6: Logical Integrity (Clicks vs Impressions)

It is impossible for the number of clicks to exceed impressions.
- Impression is when something appears on the screen of the user.
- Click is when that user clicks something clickable.

In [161]:
# removing duplicated column "clicks"
df = df.loc[:, ~df.columns.duplicated()]

In [162]:
impossible_mask = df["clicks"] > df["impressions"]

display(df.loc[impossible_mask, ["campaign_id", "impressions", "clicks"]].head())

,campaign_id,impressions,clicks


There are no instances where clicks are higher than impressions.

## Step 7: Logical Integrity (Time Travel)

In [163]:
time_travel_mask = df["end_date"] < df["start_date"]

print(f"Total records with end date preceding start date: {time_travel_mask.sum()}")
display(df.loc[time_travel_mask, ["campaign_id", "start_date", "end_date"]].head())

Total records with end date preceeding start date: 110


,campaign_id,start_date,end_date
23,CMP-00024,2023-05-06,2023-05-01
54,CMP-00055,2023-09-01,2023-08-27
71,CMP-00072,2023-02-01,2023-01-27
86,CMP-00087,2023-10-06,2023-07-08
97,CMP-00098,2023-07-02,2023-02-26


In [185]:
# swapping the values for the 110 records where end_date precedes start_date
df.loc[time_travel_mask, ["start_date", "end_date"]] = df.loc[time_travel_mask, ["end_date", "start_date"]].values

## Step 8: Handling Outliers (Winsorizing)

In [187]:
df.describe()

,start_date,end_date,impressions,clicks,spend,conversions
count,2020,2020,2020.000000,2020.000000,2020.000000,1820.000000
mean,2023-07-04 02:49:39.801980,2023-07-26 10:27:19.603960,49839.896040,1500.744059,3094.844970,186.085714
min,2022-12-31 00:00:00,2023-01-05 00:00:00,1055.000000,11.000000,-2503.310000,0.000000
25%,2023-04-04 00:00:00,2023-05-01 00:00:00,25033.500000,650.750000,649.997500,68.000000
50%,2023-07-03 00:00:00,2023-07-28 00:00:00,50097.500000,1245.000000,1427.105000,142.000000
75%,2023-10-05 06:00:00,2023-10-26 00:00:00,74784.250000,2185.250000,2638.382500,257.000000
max,2024-01-01 00:00:00,2024-01-30 00:00:00,99875.000000,4812.000000,500000.000000,943.000000
std,NaN,NaN,28579.637473,1084.765654,24811.020701,160.129172


In [212]:
Q1 = df["spend"].quantile(0.25)
Q3 = df["spend"].quantile(0.75)

IQR = Q3 - Q1
upper_limit = Q3 + (3 * IQR)

print(f"upper_limit = {upper_limit}")

outlier_mask = df["spend"] > upper_limit

display(df.loc[outlier_mask, ["campaign_id", "spend"]])

proper_upper_limit = df.loc[outlier_mask, "spend"].min()

print(f"proper_upper_limit = {proper_upper_limit}")

df.loc[outlier_mask, "spend"] = proper_upper_limit

display(df.loc[outlier_mask, ["campaign_id", "spend"]])

upper_limit = 8603.537499999999


,campaign_id,spend
789,CMP-00790,500000.00
1443,CMP-01444,8921.51
1460,CMP-01461,500000.00
1718,CMP-01719,500000.00
1754,CMP-01755,500000.00
1781,CMP-01782,500000.00


proper_upper_limit = 8921.51


,campaign_id,spend
789,CMP-00790,8921.51
1443,CMP-01444,8921.51
1460,CMP-01461,8921.51
1718,CMP-01719,8921.51
1754,CMP-01755,8921.51
1781,CMP-01782,8921.51


In [229]:
negative_mask = df["spend"] < 0

display(df.loc[negative_mask, ["campaign_id", "spend"]])

df.loc[negative_mask, "spend"] = df.loc[negative_mask, "spend"].abs()

display(df.loc[negative_mask, ["campaign_id", "spend"]])

,campaign_id,spend
144,CMP-00145,-1407.54
407,CMP-00408,-1214.52
576,CMP-00577,-821.80
604,CMP-00605,-942.80
676,CMP-00677,-1640.88
817,CMP-00818,-1261.81
886,CMP-00887,-2162.50
924,CMP-00925,-648.30
1125,CMP-01126,-122.80
1142,CMP-01143,-20.05


,campaign_id,spend
144,CMP-00145,1407.54
407,CMP-00408,1214.52
576,CMP-00577,821.80
604,CMP-00605,942.80
676,CMP-00677,1640.88
817,CMP-00818,1261.81
886,CMP-00887,2162.50
924,CMP-00925,648.30
1125,CMP-01126,122.80
1142,CMP-01143,20.05


## Step 9: String Parsing (Feature Extraction)

The regex `Q\d_([^_]+)_` searches for everything between the first and the second occurrences of an underscore.

In [222]:
display(df[["campaign_id", "campaign_name"]].head())

df["season"] = df["campaign_name"].str.extract(r"Q\d_([^_]+)_")

display(df[["campaign_id", "campaign_name", "season"]].head())

,campaign_id,campaign_name
0,CMP-00001,Q4_Summer_CMP-00001
1,CMP-00002,Q1_Launch_CMP-00002
2,CMP-00003,Q3_Winter_CMP-00003
3,CMP-00004,Q1_BlackFriday_CMP-00004
4,CMP-00005,Q2_Winter_CMP-00005


,campaign_id,campaign_name,season
0,CMP-00001,Q4_Summer_CMP-00001,Summer
1,CMP-00002,Q1_Launch_CMP-00002,Launch
2,CMP-00003,Q3_Winter_CMP-00003,Winter
3,CMP-00004,Q1_BlackFriday_CMP-00004,BlackFriday
4,CMP-00005,Q2_Winter_CMP-00005,Winter
